# R02 — One frozen candidate table, one pipeline, several seeds: a self-consistent re-confirmation

**GPU required (≈ 2–4 h per dataset with 3 seeds; `REVIEW_SMOKE=1` = plumbing check; set `REVIEW_SEEDS` to change the number of seeds).**
Needs `MVTEC_ROOT` and `PDT_ROOT`. Resumable: every PatchCore run is cached under `review/outputs/cache/`.

### Why
`R00` found that the repository's confirmation artefacts do **not** agree with each other: the review budget of the
DiCE/ACFX candidates in the committed candidate table differs from the budget stored in `captured_scores/*.npz` and in
`decomposition_per_candidate_*.csv` for 14/24 (MVTec) and 13/24 (Proditec) candidates, and the largest absolute difference between
the “rerun” and the original confirmation F1 is 0.07 (MVTec) / 0.15 (Proditec) — not the 0.019 stated in the paper.
That is not seed noise; it points to artefacts created from *different generations* of the stochastic candidate table.

This notebook removes the ambiguity. It
1. **freezes** the committed 24-row candidate table (hash recorded) and never regenerates it,
2. re-trains every candidate with the **same code path** under **N seeds** (seed changes train sub-sample, patch sub-sample and coreset),
3. re-trains a stratified sample of **logged HPO trials** (a noise floor: how well does the pipeline reproduce a value it already logged?),
4. reports, for each candidate, the surrogate prediction, the recorded value (where one exists), the validation F1 (the quantity the
   surrogate was trained on), the held-out final F1, and their differences, with seed-to-seed standard deviations.

Protocol per dataset: **MVTec** = one PatchCore + one review-budget threshold per category, mean over the four categories (the
protocol that generated the surrogate’s target); the pooled variant of the same models is also stored. **Proditec** = one model.
All numbers below replace both `*_patchcore_evaluation.csv` and `decomposition_per_candidate_*.csv`.

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# C1) GPU / dataset configuration  <-- EDIT THIS CELL
# ============================================================
import os, time, pickle

# Local copies of the datasets (the split CSVs hold absolute paths of the original server and are
# re-rooted onto these folders).  Environment variables take precedence.
#   MVTEC_ROOT : folder that contains bottle/ cable/ capsule/ hazelnut/ (MVTec AD)
#   PDT_ROOT   : folder that contains train/ and test/ (Proditec public dataset; proprietary)
MVTEC_ROOT = os.environ.get("MVTEC_ROOT", "")      # e.g. "/data/mvtec_anomaly_detection"
PDT_ROOT = os.environ.get("PDT_ROOT", "")          # e.g. "/data/pdt/dataset_public"
DEVICE_NAME = os.environ.get("REVIEW_DEVICE", "")  # "" = auto (cuda if available), or "cuda:1" / "cpu"
SMOKE = os.environ.get("REVIEW_SMOKE", "0") == "1" # 1 = tiny run (2-3 items) to check the plumbing
NUM_WORKERS = int(os.environ.get("REVIEW_WORKERS", "2"))

sys.path.insert(0, str(REVIEW))
import torch
import patchcore_review_lib as L
L.NUM_WORKERS = NUM_WORKERS
L.set_device(DEVICE_NAME or None)
torch.manual_seed(SEED)
if SMOKE:   # plumbing runs never touch the real outputs / cache
    OUT = OUT / "smoke"
    OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
    for _d in (OUT_TABLES, OUT_FIGS):
        _d.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "cache"
CACHE.mkdir(exist_ok=True)
print("device:", L.DEVICE, "| smoke:", SMOKE)
print("MVTEC_ROOT:", MVTEC_ROOT or "(unset)", "| PDT_ROOT:", PDT_ROOT or "(unset)")


def need_images(df: pd.DataFrame, what: str):
    rate = L.path_exist_rate(df)
    print(f"{what}: {len(df)} images, path_exists_rate={rate:.3f}")
    if rate < 1.0:
        raise FileNotFoundError(f"{what}: some images are missing. Set MVTEC_ROOT / PDT_ROOT "
                                "(see the configuration cell).")


def load_data(ds: str):
    """ds in {'mvtec' (per-category dict), 'mvtec_pooled' (one pooled frame set), 'proditec'}."""
    if ds.startswith("mvtec"):
        cd = L.load_mvtec_class_data(REPO / "mvtec" / "mvtec_splits", MVTEC_ROOT or None)
        if SMOKE:
            cd = {c: L.smoke_subsample(v, 14) for c, v in cd.items()}
        data = L.pool_class_data(cd) if ds == "mvtec_pooled" else cd
        if ds == "mvtec":
            for c, v in cd.items():
                need_images(pd.concat([v["train"], v["val"], v["final"]]), f"MVTec/{c}")
        else:
            need_images(pd.concat([data["train"], data["val"], data["final"]]), "MVTec pooled")
        return data
    data = L.load_proditec_data(REPO / "proditec" / "patchcore_splits", PDT_ROOT or None)
    if SMOKE:
        data = L.smoke_subsample(data, 40)
    need_images(pd.concat([data["train"], data["val"], data["final"]]), "Proditec")
    return data

In [ ]:
# ============================================================
# R2.1) Configuration, frozen table, surrogate
# ============================================================
import hashlib
from catboost import CatBoostRegressor, Pool

N_SEEDS = int(os.environ.get("REVIEW_SEEDS", "3"))
N_LOGGED = int(os.environ.get("REVIEW_N_LOGGED", "40"))          # logged HPO trials re-trained per dataset (stratified over the F1 range)
N_TOP = int(os.environ.get("REVIEW_N_TOP", "8"))                 # plus the best-N logged trials (the 'upper-quantile' regime)
DATASETS = ["mvtec", "proditec"]
if SMOKE:
    N_SEEDS, N_LOGGED, N_TOP = 2, 3, 1
SEEDS = tuple(SEED + i for i in range(N_SEEDS))
print("seeds:", SEEDS, "| logged trials per dataset:", N_LOGGED, "+ top", N_TOP)

HPO_DB = {"mvtec": (REPO / "mvtec" / "outputs_patchcore" / "mvtec_4class_patchcore_hpo_logs.db", None),
          "proditec": (REPO / "proditec" / "outputs_patchcore" / "pdt_public_patchcore_hpo_logs.db", "v1.0.1")}
FROZEN, SURR, LOGGED = {}, {}, {}
for ds in DATASETS:
    s = DS[ds]
    tab = pd.read_csv(s["cfs"])
    FROZEN[ds] = tab
    meta = json.loads(s["meta"].read_text())
    mdl = CatBoostRegressor(); mdl.load_model(str(s["model"]))
    SURR[ds] = (mdl, meta)
    h = hashlib.md5(s["cfs"].read_bytes()).hexdigest()
    print(f"{ds:9s} frozen table md5 {h[:12]}  rows {len(tab)}  methods {tab.method.value_counts().to_dict()}")
    db, nm = HPO_DB[ds]
    LOGGED[ds] = L.load_hpo_trials(db, nm)


def surrogate_predict(ds, frame: pd.DataFrame) -> np.ndarray:
    """Saved CatBoost surrogate on a frame with or without the ``params_`` prefix."""
    mdl, meta = SURR[ds]
    X = pd.DataFrame({f: frame[f].values if f in frame else frame[f.replace("params_", "")].values for f in meta["features"]})
    for c in meta["categorical_features"]:
        X[c] = X[c].astype(str).str.replace(r"\.0$", "", regex=True)      # '8.0' -> '8' as in the training frame
    return np.asarray(mdl.predict(X[list(mdl.feature_names_)]), float)


# sanity: the surrogate reproduces its own stored prediction for the frozen table
for ds in DATASETS:
    p = surrogate_predict(ds, FROZEN[ds])
    print(f"{ds:9s} max|fresh surrogate - stored surrogate_pred| over the 24 candidates = {np.abs(p - FROZEN[ds].surrogate_pred.values).max():.2e}")

In [ ]:
# ============================================================
# R2.2) Stratified sample of logged trials (a noise floor for the pipeline)
# ============================================================
def pick_logged(ds, n_strat, n_top):
    t = LOGGED[ds].dropna(subset=["value"]).sort_values("value").reset_index(drop=True)
    if "soft_review_budget" in t:
        t = t[t.soft_review_budget.astype(float) > 0]
    q = np.linspace(0, len(t) - 1 - n_top, n_strat).round().astype(int) if n_strat else []
    top = t.index[-n_top:] if n_top else []
    idx = sorted(set(list(q) + list(top)))
    sel = t.loc[idx].copy()
    sel["stratum"] = ["top" if i in set(top) else "stratified" for i in idx]
    return sel


PICK = {ds: pick_logged(ds, N_LOGGED, N_TOP) for ds in DATASETS}
for ds in DATASETS:
    p = PICK[ds]
    print(f"{ds:9s} logged trials re-trained: {len(p)} | logged value range {p.value.min():.3f}-{p.value.max():.3f} | "
          f"pre_trained=False: {(~p.pre_trained.astype(bool)).sum()}")
    save_table(p.drop(columns=[c for c in ("trial_id", "state") if c in p]), f"R02_{ds}_logged_trial_selection")

In [ ]:
# ============================================================
# R2.3) Run everything (resumable)
# ============================================================
RES = {}
for ds in DATASETS:
    data = load_data(ds)
    cands = FROZEN[ds]
    if SMOKE:
        cands = cands.head(3)
    print(f"\n=== {ds}: {len(cands)} frozen candidates x {len(SEEDS)} seeds ===")
    tc, rawc = L.run_many(ds, cands.to_dict("records"), data, seeds=SEEDS, cache_dir=CACHE, ids=cands.cf_id.tolist(),
                          offsets=cands.cf_id.tolist())
    tc["kind"] = "candidate"
    p = PICK[ds]
    print(f"\n=== {ds}: {len(p)} logged trials x {len(SEEDS)} seeds ===")
    cfgs = p.to_dict("records")
    tl, rawl = L.run_many(ds, cfgs, data, seeds=SEEDS, cache_dir=CACHE, ids=[f"trial{int(n)}" for n in p.number],
                          offsets=[int(n) for n in p.number])
    tl["kind"] = "logged"
    RES[ds] = dict(cand=tc, logged=tl, rawc=rawc, rawl=rawl, data=data)

In [ ]:
# ============================================================
# R2.4) Candidate table: surrogate vs re-measured, per seed and aggregated
# ============================================================
AGG = {}
for ds in DATASETS:
    tc = RES[ds]["cand"].copy()
    fr = FROZEN[ds].rename(columns={"cf_id": "id"})
    tc = tc.merge(fr[["id", "method", "surrogate_pred", "distance_to_factual", "params_soft_review_budget"]], on="id")
    tc["err_val"] = tc.f1_val - tc.surrogate_pred
    tc["err_test"] = tc.f1_test - tc.surrogate_pred
    g = (tc.groupby(["id", "method"]).agg(budget=("params_soft_review_budget", "first"), surrogate=("surrogate_pred", "first"),
                                          f1_val=("f1_val", "mean"), f1_val_sd=("f1_val", "std"),
                                          f1_test=("f1_test", "mean"), f1_test_sd=("f1_test", "std"),
                                          auroc_test=("auroc_test", "mean"), err_val=("err_val", "mean"), err_test=("err_test", "mean"),
                                          runtime=("runtime_sec", "mean")).reset_index())
    AGG[ds] = g
    save_table(g, f"R02_{ds}_candidates_aggregated")
    save_table(tc, f"R02_{ds}_candidates_per_seed")
    m = (g.groupby("method").agg(n=("id", "size"), surrogate=("surrogate", "mean"), f1_val=("f1_val", "mean"), f1_test=("f1_test", "mean"),
                                 auroc_test=("auroc_test", "mean"), err_val=("err_val", "mean"), err_test=("err_test", "mean"),
                                 seed_sd_test=("f1_test_sd", "mean")).reset_index())
    al = g[["surrogate", "f1_val", "f1_test", "auroc_test", "err_val", "err_test", "f1_test_sd"]].mean()
    m = pd.concat([m, pd.DataFrame([dict(method="ALL", n=len(g), surrogate=al.surrogate, f1_val=al.f1_val, f1_test=al.f1_test,
                                         auroc_test=al.auroc_test, err_val=al.err_val, err_test=al.err_test, seed_sd_test=al.f1_test_sd)])])
    print(f"\n[{ds}] frozen-table confirmation, mean over {len(SEEDS)} seeds"); display(m.round(4))
    save_table(m, f"R02_{ds}_method_summary")

In [ ]:
# ============================================================
# R2.5) Noise floor: logged value vs re-measured value; and what the surrogate says about the same trials
# ============================================================
NF = {}
for ds in DATASETS:
    tl = RES[ds]["logged"].copy()
    p = PICK[ds].assign(id=[f"trial{int(n)}" for n in PICK[ds].number])
    tl = tl.merge(p[["id", "value", "stratum", "pre_trained", "soft_review_budget"]].rename(columns={"value": "logged_value"}), on="id")
    tl["pre_trained"] = tl.pre_trained.astype(bool)
    tl["surrogate"] = tl.id.map(dict(zip(p.id, surrogate_predict(ds, p))))
    tl["diff_val"] = tl.f1_val - tl.logged_value
    g = tl.groupby(["id", "stratum", "pre_trained"]).agg(logged=("logged_value", "first"), surrogate=("surrogate", "first"),
                                                         f1_val=("f1_val", "mean"), f1_val_sd=("f1_val", "std"),
                                                         f1_test=("f1_test", "mean"), diff_val=("diff_val", "mean")).reset_index()
    NF[ds] = g
    save_table(g, f"R02_{ds}_logged_rerun")
    rows = []
    for name, sub in (("all", g), ("pre_trained=True", g[g.pre_trained]), ("pre_trained=False", g[~g.pre_trained]),
                      ("top", g[g.stratum == "top"])):
        if len(sub) == 0:
            continue
        rows.append(dict(subset=name, n=len(sub), mean_abs_diff=sub.diff_val.abs().mean(), max_abs_diff=sub.diff_val.abs().max(),
                         mean_signed_diff=sub.diff_val.mean(), seed_sd=sub.f1_val_sd.mean(),
                         surrogate_mae_vs_rerun=(sub.surrogate - sub.f1_val).abs().mean(),
                         surrogate_bias_vs_rerun=(sub.surrogate - sub.f1_val).mean()))
    nf = pd.DataFrame(rows)
    print(f"\n[{ds}] logged value vs re-measured validation F1 (pipeline noise floor)"); display(nf.round(4))
    save_table(nf, f"R02_{ds}_noise_floor")

In [ ]:
# ============================================================
# R2.6) Candidates vs logged trials at the same review budget  (is a candidate 'special'?)
# ============================================================
rows = []
for ds in DATASETS:
    g, nf = AGG[ds], NF[ds]
    rows.append(dict(dataset=ds, group="candidates (frozen table)", n=len(g), surrogate=g.surrogate.mean(), f1_val=g.f1_val.mean(),
                     f1_test=g.f1_test.mean(), err_val=g.err_val.mean(), err_test=g.err_test.mean()))
    top = nf[nf.stratum == "top"]
    if len(top):
        rows.append(dict(dataset=ds, group="top logged trials", n=len(top), surrogate=top.surrogate.mean(), f1_val=top.f1_val.mean(),
                         f1_test=top.f1_test.mean(), err_val=(top.f1_val - top.surrogate).mean(), err_test=(top.f1_test - top.surrogate).mean()))
    st = nf[nf.stratum == "stratified"]
    rows.append(dict(dataset=ds, group="stratified logged trials", n=len(st), surrogate=st.surrogate.mean(), f1_val=st.f1_val.mean(),
                     f1_test=st.f1_test.mean(), err_val=(st.f1_val - st.surrogate).mean(), err_test=(st.f1_test - st.surrogate).mean()))
CMP = pd.DataFrame(rows); display(CMP.round(4)); save_table(CMP, "R02_candidates_vs_logged_trials")
print("\nIf the top *logged* trials show a similar (re-measured - surrogate) shift as the generated candidates, the gap is a property of the "
      "upper-quantile region of the surrogate (selection-conditioned fidelity), not of the counterfactual generator.")

In [ ]:
# ============================================================
# R2.7) Comparison with the numbers currently in the paper
# ============================================================
rows = []
for ds in DATASETS:
    s = DS[ds]
    ev = pd.read_csv(s["eval"]); dc = pd.read_csv(s["decomp"])
    g = AGG[ds].set_index("id")
    ev_f1 = ev.set_index("cf_id").f1_test; dc_f1 = dc.set_index("cf_id").f1_test_deployed
    for name, ref in (("evaluation CSV (Table 4 source)", ev_f1), ("decomposition 'rerun' CSV", dc_f1)):
        d = (g.f1_test - ref.reindex(g.index))
        rows.append(dict(dataset=ds, committed_artefact=name, committed_mean=ref.reindex(g.index).mean(), frozen_rerun_mean=g.f1_test.mean(),
                         mean_abs_diff=d.abs().mean(), max_abs_diff=d.abs().max(), n_gt_0p02=int((d.abs() > 0.02).sum()),
                         seed_sd_mean=g.f1_test_sd.mean()))
PV = pd.DataFrame(rows); display(PV.round(4)); save_table(PV, "R02_committed_vs_frozen_rerun")

In [ ]:
# ============================================================
# R2.8) Figures
# ============================================================
col = {"DiCE": "#3b82d4", "CFNOW": "#7c5cd8", "ACFX": "#d98a2b"}
fig, ax = plt.subplots(1, 2, figsize=WIDE_COL)
for a, ds in zip(ax, DATASETS):
    g, nf = AGG[ds], NF[ds]
    for m, gg in g.groupby("method"):
        a.errorbar(gg.surrogate, gg.f1_test, yerr=gg.f1_test_sd.fillna(0), fmt="o", ms=3, lw=.6, color=col[m], label=m)
    top = nf[nf.stratum == "top"]
    a.scatter(top.surrogate, top.f1_test, marker="s", s=14, facecolor="none", edgecolor="k", label="top logged")
    lo = min(g.surrogate.min(), g.f1_test.min(), nf.surrogate.min()) - .03; hi = max(g.surrogate.max(), nf.surrogate.max(), g.f1_test.max()) + .03
    a.plot([lo, hi], [lo, hi], "k:", lw=.7); a.set_xlim(lo, hi); a.set_ylim(lo, hi)
    a.set_xlabel("Surrogate prediction"); a.set_ylabel("Held-out F1 (mean ± SD over seeds)"); a.set_title(ds.upper() if ds == "mvtec" else "Proditec")
ax[0].legend(frameon=False, fontsize=6, loc="upper left")
fig.tight_layout(); save_fig(fig, "R02_frozen_table_confirmation")

fig, ax = plt.subplots(1, 2, figsize=WIDE_COL)
for a, ds in zip(ax, DATASETS):
    nf = NF[ds]
    a.scatter(nf.logged, nf.f1_val, s=10, c=np.where(nf.pre_trained, "#3b82d4", "#d98a2b"))
    lo, hi = nf.logged.min() - .02, nf.logged.max() + .02
    a.plot([lo, hi], [lo, hi], "k:", lw=.7); a.set_xlabel("Logged HPO value"); a.set_ylabel("Re-measured validation F1")
    a.set_title(("MVTec" if ds == "mvtec" else "Proditec") + " (orange: pre_trained = False)")
fig.tight_layout(); save_fig(fig, "R02_pipeline_noise_floor")

In [ ]:
# ============================================================
# R2.9) Freeze numbers
# ============================================================
merge_numbers("R02_frozen_table", dict(
    smoke=SMOKE, seeds=list(SEEDS),
    frozen_table_md5={ds: hashlib.md5(DS[ds]["cfs"].read_bytes()).hexdigest() for ds in DATASETS},
    summary={ds: pd.read_csv(OUT_TABLES / f"R02_{ds}_method_summary.csv").set_index("method").round(4).to_dict(orient="index") for ds in DATASETS},
    noise_floor={ds: pd.read_csv(OUT_TABLES / f"R02_{ds}_noise_floor.csv").set_index("subset").round(4).to_dict(orient="index") for ds in DATASETS},
    committed_vs_rerun=PV.round(4).to_dict(orient="records"),
))